Q401 Retail Commerce Operations

In [1]:
from typing import List
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, to_date, datediff, when, sum, avg, count



In [2]:
def define_schema() -> StructType:
    return StructType([
        StructField("txn_id",StringType(),True),
        StructField("order_date",StringType(),True),
        StructField("customer_id",StringType(),True),
        StructField("region",StringType(),True),
        StructField("channel",StringType(),True),
        StructField("category",StringType(),True),
        StructField("product_id",StringType(),True),
        StructField("quantity",IntegerType(),True),
        StructField("unit_price",DoubleType(),True),
        StructField("discount_rate",DoubleType(),True),
        StructField("returned",StringType(),True),
        StructField("ship_date",StringType(),True),
        StructField("delivery_date",StringType(),True)
    ])

def load_data(spark: SparkSession, path: str, schema: StructType) -> DataFrame:
    return spark.read.csv(path,header=True,schema=schema)

def parse_dates(df: DataFrame) -> DataFrame:
    return (
        df.withColumn("order_date",to_date("order_date"))
        .withColumn("ship_date",to_date("ship_date"))
        .withColumn("delivery_date",to_date("delivery_date"))
    )

def add_gross_amount(df: DataFrame) -> DataFrame:
    return df.withColumn("gross_amount",col("quantity")*col("unit_price"))


def add_net_amount(df: DataFrame) -> DataFrame:
    df=add_gross_amount(df)
    return df.withColumn("net_amount",col("gross_amount")*(1-col("discount_rate")))

def add_delivery_days(df: DataFrame) -> DataFrame:
    return df.withColumn("delivery_days",datediff(col("delivery_date"),col("ship_date")))


def flag_on_time_delivery(df: DataFrame, max_days: int) -> DataFrame:
    df=add_delivery_days(df)
    return df.withColumn("is_on_time",when(col("delivery_days") <= max_days,True).otherwise(False))

def filter_returned_orders(df):
    return df.filter(col("returned")=='Y')

def filter_by_region(df, region) -> DataFrame:
    return df.filter(col("region")==region)

def top_n_customers_by_spend(df, n) -> DataFrame:
    df=add_net_amount(df)
    return (df.groupBy(col("customer_id"))
            .agg(sum(col("net_amount")).alias("total_spend"))
            .orderBy(col("total_spend").desc(),col("customer_id").asc())).limit(n)

def revenue_by_category(df):
    df=add_net_amount(df)
    return df.groupBy("category").agg(sum("net_amount").alias("total_revenue"))

def top_category_by_revenue(df) -> str:
    df=add_net_amount(df)
    row=df.groupBy("category").agg(sum("net_amount").alias("total_revenue")).orderBy(["total_revenue","category"],ascending=[0,1]).limit(1).collect()
    return row[0]["category"] if row else None

def avg_discount_by_channel(df):
  return df.groupby("channel").agg(avg("discount_rate").alias("avg_discount"))

def daily_revenue_trend(df):
  df=add_net_amount(df)
  return df.groupby("order_date").agg(sum("net_amount").alias("daily_revenue")).orderBy("order_date")
def high_value_orders(df, threshold):
  df=add_net_amount(df)
  return df.filter(col("net_amount")>threshold)
def count_late_deliveries(df, max_days):
  df=add_delivery_days(df)
  return df.filter(col("delivery_days")>max_days).count()
def return_rate_by_category(df):
  df=df.groupby("category").agg(count("*").alias("total_cnt"),sum(when(col("returned")=="Y",1).otherwise(0)).alias("returned_cnt"))
  return df.withColumn("return_rate",col("returned_cnt")/col("total_cnt"))

def best_selling_product(df):
  pass
def list_channels(df)->List[str]:
  df=df.select("channel").distinct().orderBy("channel")
  return df.collect()
def orders_in_date_range(df, start_date, end_date):
  return df.filter((col("order_date")>=start_date) &(col("order_date")<=end_date))

In [3]:
# Create Spark session
spark = SparkSession.builder \
    .appName("Retail Commerce Operations") \
    .getOrCreate()


# 1. Define schema
schema = define_schema()


# 2. Load data
df = load_data(
    spark,
    "/content/commerce.csv",
    schema
)

print("Original Data")
df.show()

Original Data
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+
|txn_id|order_date|customer_id|region|channel|   category|product_id|quantity|unit_price|discount_rate|returned| ship_date|delivery_date|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+
| T0001|2025-01-02|       C001| North|    App|    Grocery|      P101|       3|     120.0|         0.05|       N|2025-01-02|   2025-01-04|
| T0002|2025-01-03|       C002| South|    Web|Electronics|      P205|       1|    1500.0|          0.1|       N|2025-01-03|   2025-01-05|
| T0003|2025-01-03|       C001| North|    App|    Grocery|      P102|       5|      60.0|          0.0|       Y|2025-01-03|   2025-01-08|
| T0004|2025-01-05|       C003|  East|  Store|       Home|      P310|       2|     800.0|         0.15|       N|2025-01-06|   2025-01-07|
| T0005|2025-01-05| 

In [4]:
# 3. Parse dates
df = parse_dates(df)

print("After Parsing Dates")
df.show()

After Parsing Dates
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+
|txn_id|order_date|customer_id|region|channel|   category|product_id|quantity|unit_price|discount_rate|returned| ship_date|delivery_date|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+
| T0001|2025-01-02|       C001| North|    App|    Grocery|      P101|       3|     120.0|         0.05|       N|2025-01-02|   2025-01-04|
| T0002|2025-01-03|       C002| South|    Web|Electronics|      P205|       1|    1500.0|          0.1|       N|2025-01-03|   2025-01-05|
| T0003|2025-01-03|       C001| North|    App|    Grocery|      P102|       5|      60.0|          0.0|       Y|2025-01-03|   2025-01-08|
| T0004|2025-01-05|       C003|  East|  Store|       Home|      P310|       2|     800.0|         0.15|       N|2025-01-06|   2025-01-07|
| T0005|2025-0

In [5]:
# 4. Add gross amount
result = add_gross_amount(df)

print("Gross Amount")
result.show()

Gross Amount
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+------------+
|txn_id|order_date|customer_id|region|channel|   category|product_id|quantity|unit_price|discount_rate|returned| ship_date|delivery_date|gross_amount|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+------------+
| T0001|2025-01-02|       C001| North|    App|    Grocery|      P101|       3|     120.0|         0.05|       N|2025-01-02|   2025-01-04|       360.0|
| T0002|2025-01-03|       C002| South|    Web|Electronics|      P205|       1|    1500.0|          0.1|       N|2025-01-03|   2025-01-05|      1500.0|
| T0003|2025-01-03|       C001| North|    App|    Grocery|      P102|       5|      60.0|          0.0|       Y|2025-01-03|   2025-01-08|       300.0|
| T0004|2025-01-05|       C003|  East|  Store|       Home|      P310|       2|   

In [6]:
# 5. Add net amount
result = add_net_amount(df)

print("Net Amount")
result.show()

Net Amount
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+------------+-----------------+
|txn_id|order_date|customer_id|region|channel|   category|product_id|quantity|unit_price|discount_rate|returned| ship_date|delivery_date|gross_amount|       net_amount|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+------------+-----------------+
| T0001|2025-01-02|       C001| North|    App|    Grocery|      P101|       3|     120.0|         0.05|       N|2025-01-02|   2025-01-04|       360.0|            342.0|
| T0002|2025-01-03|       C002| South|    Web|Electronics|      P205|       1|    1500.0|          0.1|       N|2025-01-03|   2025-01-05|      1500.0|           1350.0|
| T0003|2025-01-03|       C001| North|    App|    Grocery|      P102|       5|      60.0|          0.0|       Y|2025-01-03|   2025-01-08|       

In [7]:
# 6. Add delivery days
result = add_delivery_days(df)

print("Delivery Days")
result.show()


Delivery Days
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+-------------+
|txn_id|order_date|customer_id|region|channel|   category|product_id|quantity|unit_price|discount_rate|returned| ship_date|delivery_date|delivery_days|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+-------------+
| T0001|2025-01-02|       C001| North|    App|    Grocery|      P101|       3|     120.0|         0.05|       N|2025-01-02|   2025-01-04|            2|
| T0002|2025-01-03|       C002| South|    Web|Electronics|      P205|       1|    1500.0|          0.1|       N|2025-01-03|   2025-01-05|            2|
| T0003|2025-01-03|       C001| North|    App|    Grocery|      P102|       5|      60.0|          0.0|       Y|2025-01-03|   2025-01-08|            5|
| T0004|2025-01-05|       C003|  East|  Store|       Home|      P310|     

In [8]:
# 7. Flag on-time delivery
result = flag_on_time_delivery(df, 5)

print("On Time Delivery")
result.show()


On Time Delivery
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+-------------+----------+
|txn_id|order_date|customer_id|region|channel|   category|product_id|quantity|unit_price|discount_rate|returned| ship_date|delivery_date|delivery_days|is_on_time|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+-------------+----------+
| T0001|2025-01-02|       C001| North|    App|    Grocery|      P101|       3|     120.0|         0.05|       N|2025-01-02|   2025-01-04|            2|      true|
| T0002|2025-01-03|       C002| South|    Web|Electronics|      P205|       1|    1500.0|          0.1|       N|2025-01-03|   2025-01-05|            2|      true|
| T0003|2025-01-03|       C001| North|    App|    Grocery|      P102|       5|      60.0|          0.0|       Y|2025-01-03|   2025-01-08|            5|      true|
| T00

In [9]:
# 8. Filter returned orders
result = filter_returned_orders(df)

print("Returned Orders")
result.show()


Returned Orders
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+
|txn_id|order_date|customer_id|region|channel|   category|product_id|quantity|unit_price|discount_rate|returned| ship_date|delivery_date|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+
| T0003|2025-01-03|       C001| North|    App|    Grocery|      P102|       5|      60.0|          0.0|       Y|2025-01-03|   2025-01-08|
| T0006|2025-01-08|       C005| South|    App|Electronics|      P206|       1|     900.0|          0.2|       Y|2025-01-08|   2025-01-10|
| T0012|2025-01-22|       C009|  East|    Web|       Home|      P312|       2|     650.0|         0.08|       Y|2025-01-22|   2025-01-29|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+



In [10]:
# 9. Filter by region
result = filter_by_region(df, "North")

print("Orders by Region")
result.show()

Orders by Region
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+
|txn_id|order_date|customer_id|region|channel|   category|product_id|quantity|unit_price|discount_rate|returned| ship_date|delivery_date|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+
| T0001|2025-01-02|       C001| North|    App|    Grocery|      P101|       3|     120.0|         0.05|       N|2025-01-02|   2025-01-04|
| T0003|2025-01-03|       C001| North|    App|    Grocery|      P102|       5|      60.0|          0.0|       Y|2025-01-03|   2025-01-08|
| T0009|2025-01-15|       C007| North|  Store|    Fashion|      P451|       3|     180.0|          0.1|       N|2025-01-15|   2025-01-18|
| T0011|2025-01-20|       C001| North|    App|Electronics|      P207|       1|    2200.0|         0.12|       N|2025-01-20|   2025-01-27|
+------+---------

In [11]:
# 10. Top N customers
result = top_n_customers_by_spend(df, 5)

print("Top 5 Customers")
result.show()


Top 5 Customers
+-----------+-----------------+
|customer_id|      total_spend|
+-----------+-----------------+
|       C002|           4200.0|
|       C001|           2578.0|
|       C003|           1360.0|
|       C009|           1196.0|
|       C004|929.9999999999999|
+-----------+-----------------+



In [12]:
# 11. Revenue by category
result = revenue_by_category(df)

print("Revenue by Category")
result.show()

Revenue by Category
+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|       Home|       2956.0|
|    Fashion|       1416.0|
|    Grocery|        892.0|
|Electronics|       6856.0|
+-----------+-------------+



In [13]:
# 12. Top category by revenue
result = top_category_by_revenue(df)

print("Top Category")
print(result)

Top Category
Electronics


In [14]:
# 13. Average discount by channel
result = avg_discount_by_channel(df)

print("Average Discount by Channel")
result.show()

Average Discount by Channel
+-------+-------------------+
|channel|       avg_discount|
+-------+-------------------+
|  Store|              0.125|
|    App|              0.074|
|    Web|0.06000000000000001|
+-------+-------------------+



In [15]:
# 14. Daily revenue trend
result = daily_revenue_trend(df)

print("Daily Revenue")
result.show()

Daily Revenue
+----------+-------------+
|order_date|daily_revenue|
+----------+-------------+
|2025-01-02|        342.0|
|2025-01-03|       1650.0|
|2025-01-05|       2290.0|
|2025-01-08|        720.0|
|2025-01-10|       2850.0|
|2025-01-12|        400.0|
|2025-01-15|        486.0|
|2025-01-18|        250.0|
|2025-01-20|       1936.0|
|2025-01-22|       1196.0|
+----------+-------------+



In [16]:
# 15. High value orders
result = high_value_orders(df, 1000)

print("High Value Orders")
result.show()

High Value Orders
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+------------+----------+
|txn_id|order_date|customer_id|region|channel|   category|product_id|quantity|unit_price|discount_rate|returned| ship_date|delivery_date|gross_amount|net_amount|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+------------+----------+
| T0002|2025-01-03|       C002| South|    Web|Electronics|      P205|       1|    1500.0|          0.1|       N|2025-01-03|   2025-01-05|      1500.0|    1350.0|
| T0004|2025-01-05|       C003|  East|  Store|       Home|      P310|       2|     800.0|         0.15|       N|2025-01-06|   2025-01-07|      1600.0|    1360.0|
| T0007|2025-01-10|       C002| South|    Web|Electronics|      P205|       2|    1500.0|         0.05|       N|2025-01-10|   2025-01-12|      3000.0|    2850.0|
| T0011|20

In [17]:
# 16. Count late deliveries
result = count_late_deliveries(df, 5)

print("Number of Late Deliveries")
print(result)

Number of Late Deliveries
2


In [18]:
# 17. Return rate by category
result = return_rate_by_category(df)

print("Return Rate by Category")
result.show()

Return Rate by Category
+-----------+---------+------------+------------------+
|   category|total_cnt|returned_cnt|       return_rate|
+-----------+---------+------------+------------------+
|       Home|        3|           1|0.3333333333333333|
|    Fashion|        2|           0|               0.0|
|    Grocery|        3|           1|0.3333333333333333|
|Electronics|        4|           1|              0.25|
+-----------+---------+------------+------------------+



In [19]:
# 18. Best selling product
result = best_selling_product(df)

print("Best Selling Product")
print(result)


Best Selling Product
None


In [20]:
# 19. List channels
result = list_channels(df)

print("Available Channels")
print(result)

Available Channels
[Row(channel='App'), Row(channel='Store'), Row(channel='Web')]


In [21]:
# 20. Orders in date range
result = orders_in_date_range(
    df,
    "2025-01-01",
    "2025-12-31"
)

print("Orders in Date Range")
result.show()

Orders in Date Range
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+
|txn_id|order_date|customer_id|region|channel|   category|product_id|quantity|unit_price|discount_rate|returned| ship_date|delivery_date|
+------+----------+-----------+------+-------+-----------+----------+--------+----------+-------------+--------+----------+-------------+
| T0001|2025-01-02|       C001| North|    App|    Grocery|      P101|       3|     120.0|         0.05|       N|2025-01-02|   2025-01-04|
| T0002|2025-01-03|       C002| South|    Web|Electronics|      P205|       1|    1500.0|          0.1|       N|2025-01-03|   2025-01-05|
| T0003|2025-01-03|       C001| North|    App|    Grocery|      P102|       5|      60.0|          0.0|       Y|2025-01-03|   2025-01-08|
| T0004|2025-01-05|       C003|  East|  Store|       Home|      P310|       2|     800.0|         0.15|       N|2025-01-06|   2025-01-07|
| T0005|2025-

In [22]:

# Stop Spark
spark.stop()